In [398]:
import numpy as np
import pandas as pd


df = pd.read_csv("bank-full.csv", sep=";")
df.dtypes

age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object

In [399]:
df.head(3)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no


In [400]:
numerical = ["age", "balance", "previous"]

categorical = [
    "job",
    "education",
    "housing",
    "loan",
    "poutcome",
]

In [401]:
from sklearn.model_selection import train_test_split

features = categorical + numerical

X = df[features].copy()
y = df["y"].map({"yes": 1, "no": 0})

assert y.notna().all()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train,
)

len(X_train), len(X_val), len(X_test)

(27126, 9042, 9043)

In [402]:
from sklearn.feature_extraction import DictVectorizer

features = categorical + numerical

train_dicts = X_train[features].to_dict(orient="records")
val_dicts = X_val[features].to_dict(orient="records")

dv = DictVectorizer(sparse=False)

X_train_encoded = dv.fit_transform(train_dicts)
X_val_encoded = dv.transform(val_dicts)

print(X_train_encoded.shape)
print(X_val_encoded.shape)

(27126, 27)
(9042, 27)


In [403]:
from sklearn.preprocessing import StandardScaler

feature_names = dv.get_feature_names_out()

numerical_mask = np.isin(feature_names, numerical)

X_train_scaled = X_train_encoded.copy()
X_val_scaled = X_val_encoded.copy()

scaler = StandardScaler()

X_train_scaled[:, numerical_mask] = scaler.fit_transform(
    X_train_encoded[:, numerical_mask]
)

X_val_scaled[:, numerical_mask] = scaler.transform(X_val_encoded[:, numerical_mask])

X_val_scaled[0]


array([-0.08411536,  0.80055364,  0.        ,  0.        ,  1.        ,
        0.        ,  0.        ,  1.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  1.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        1.        ,  0.        ,  0.        ,  0.        ,  1.        ,
        0.        ,  0.72422931])

In [404]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver="lbfgs")

model.fit(X_train_scaled, y_train)

model.intercept_[0]

model.coef_[0].round(3)

probabilities = model.predict_proba(X_val_scaled)[:, 1]

churn_decision = probabilities >= 0.5

accuracy_score_model = (y_val == churn_decision).mean()

In [405]:
baseline_probability = y_train.mean()

baseline_probabilities = np.full(
    len(y_val),
    baseline_probability,
)

baseline_decisions = baseline_probabilities >= 0.5
baseline_decisions

accuracy_score_baseline = (y_val == baseline_decisions).mean()
X_val_scaled[baseline_decisions]

array([], shape=(0, 27), dtype=float64)

In [406]:
from sklearn.metrics import log_loss

model_loss = log_loss(y_val, probabilities)
baseline_loss = log_loss(y_val, baseline_probabilities)

In [407]:
data = {
    "Способ": ["Baseline", "Model"],
    "Accuracy": [accuracy_score_baseline, accuracy_score_model],
    "Log Loss": [baseline_loss, model_loss],
}


df_score = pd.DataFrame(data)

df_score

,Способ,Accuracy,Log Loss
0,Baseline,0.882990,0.360924
1,Model,0.894382,0.316309


Baseline получает высокую accuracy потому что:
  1. Ответ кроется в данных. Колл-во таргет данных равных yes = 1 - 0.88, то есть в нашем случаи ответы. Так как колл-во прогнозируемых ответов - yes == 0.12, а мы выдали всем значения ~0.12. То по нашему решению, когда все решение меньше 0.5 выдают - нет, будет как раз 0.12 исходя из чисто наших правил. 

In [408]:
print("Accuracy baseline: ", accuracy_score_baseline)

Accuracy baseline:  0.8829904888299049


В log_loss нужно передавать вероятности потому что:
  1. log_loss оценивает не решение, а саму вероятность. То есть, в нашем случаи, log_loss оценивает не то, сколько ответов правильно, а то на сколько близко модель подошла к правильному ответу

В данном случаи нам позволяет заключить сравнение только на основе ответов, так как log_loss к baseline мы не можем адекватно применить. *Можем, но это будет просто показывать отношение ответов друг к другу и не даст результата, хоть фактически он и может быть получен.

In [409]:
X_full_train = pd.concat([X_train, X_val])
y_full_train = pd.concat([y_train, y_val])

len(X_full_train)

36168

In [410]:
from sklearn.feature_extraction import DictVectorizer

train_dicts = X_full_train[features].to_dict(orient="records")
test_dicts = X_test[features].to_dict(orient="records")

dv = DictVectorizer(sparse=False)

X_train_encoded = dv.fit_transform(train_dicts)
X_test_encoded = dv.transform(test_dicts)


In [411]:
from sklearn.preprocessing import StandardScaler

feature_names = dv.get_feature_names_out()

numerical_mask = np.isin(feature_names, numerical)

X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()

scaler = StandardScaler()

X_train_scaled[:, numerical_mask] = scaler.fit_transform(
    X_train_encoded[:, numerical_mask]
)

X_test_scaled[:, numerical_mask] = scaler.transform(X_test_encoded[:, numerical_mask])


In [412]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver="lbfgs")

model.fit(X_train_scaled, y_full_train)

model.intercept_[0]

model.coef_[0].round(3)


probabilities = model.predict_proba(X_test_scaled)[:, 1]


churn_decision = probabilities >= 0.5

accuracy_score_model = (y_test == churn_decision).mean()



In [413]:
baseline_probability = y_full_train.mean()

baseline_probabilities = np.full(
    len(y_test),
    baseline_probability,
)

baseline_decisions = baseline_probabilities >= 0.5
baseline_decisions

accuracy_score_baseline = (y_test == baseline_decisions).mean()
X_test_scaled[baseline_decisions]


array([], shape=(0, 27), dtype=float64)

In [414]:
from sklearn.metrics import log_loss

model_loss = log_loss(y_test, probabilities)
baseline_loss = log_loss(y_test, baseline_probabilities)


In [415]:
data = {
    "Способ": ["Baseline", "Model"],
    "Accuracy": [accuracy_score_baseline, accuracy_score_model],
    "Log Loss": [baseline_loss, model_loss],
}


df_score = pd.DataFrame(data)

df_score


,Способ,Accuracy,Log Loss
0,Baseline,0.883003,0.360898
1,Model,0.893066,0.316915


In [416]:
client = X_test_scaled[0]
actual = y_test.iloc[0]

model.predict_proba(client)[:, 1]

ValueError: Expected 2D array, got 1D array instead:
array=[-0.08403175 -0.23643251  1.          0.          0.          0.
  0.          1.          0.          1.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          1.          0.          0.
  0.          1.         -0.24150867].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.